# 1. Dataset Loading and Initial Inspection

This notebook performs text preprocessing on the CivicSense
supervised dataset and the multilingual prediction pool.

The objective is to remove unnecessary formatting variations
while preserving the original meaning and multilingual content.

The original text will be retained separately for comparison
and validation.

In [1]:
# Import libraries required for text preprocessing
import pandas as pd
import numpy as np
import re
import html
import unicodedata

# Load the final supervised dataset prepared in Notebook 02
supervised_df = pd.read_csv(
    "../data/interim/civicsense_supervised_dataset.csv"
)

# Load the unlabelled multilingual complaints reserved for prediction
prediction_df = pd.read_csv(
    "../data/interim/civicsense_prediction_pool.csv"
)

# Display the dimensions of both datasets
print("Supervised Dataset Shape:", supervised_df.shape)
print("Prediction Pool Shape:", prediction_df.shape)

# Inspect the first few supervised complaints
display(supervised_df.head())

# Inspect the first few multilingual prediction complaints
display(prediction_df[["description", "detectedLanguage"]].head())

Supervised Dataset Shape: (16519, 3)
Prediction Pool Shape: (27686, 13)


,model_text,target_category,source_dataset
0,Sadak paar karne wala concrete ka puliya adhur...,Road Infrastructure,Dataset_A_Multilingual
1,"सीएमसी का पानी जारी नहीं किया जा रहा है, वे बा...",Water Supply,Dataset_A_Multilingual
2,Garbage dump all over,Waste Management,Dataset_A_Multilingual
3,Puttenahalli jheel road jo Arekere ki or jaati...,Road Infrastructure,Dataset_A_Multilingual
4,Desilted Mud is not cleared making the road di...,Waste Management,Dataset_A_Multilingual


,description,detectedLanguage
0,"Sir, we are facing lot of water problem in 8th...",English
1,"Doddabommasandra Arch se Nanjappa Circle, Nanj...",Hinglish
2,Kachra kachra kachra,Hinglish
3,Garbage dumped next to mysore circle bus stop....,English
4,Too many pot holes near GIIS school and petrol...,English


In [2]:
# Check missing values in the supervised text column
print("Supervised text missing values:",
      supervised_df["model_text"].isna().sum())

# Check missing values in the prediction pool description column
print("Prediction text missing values:",
      prediction_df["description"].isna().sum())

# Inspect the data types of the relevant text columns
print("\nSupervised text data type:")
print(supervised_df["model_text"].dtype)

print("\nPrediction text data type:")
print(prediction_df["description"].dtype)

# Display sample complaints to understand their original formatting
print("\nSupervised Dataset Examples:")
for text in supervised_df["model_text"].head(5):
    print(repr(text))

print("\nPrediction Pool Examples:")
for text in prediction_df["description"].head(5):
    print(repr(text))

Supervised text missing values: 0
Prediction text missing values: 0

Supervised text data type:
str

Prediction text data type:
str

Supervised Dataset Examples:
'Sadak paar karne wala concrete ka puliya adhura hai aur vaahano ke sugam aavagaman ke liye ramp theek se nahi banaaye gaye hain, isse har vaahan ke nichle hisse ko nuksaan pahunch raha hai aur har baa'
'सीएमसी का पानी जारी नहीं किया जा रहा है, वे बाबूसापाल्या फ्लावर गार्डन बैंगलोर में प्रति घर 500 रुपये की मांग कर रहे हैं। कार्यकर्ता ने कोई कोविड-19 निवारक उपाय नहीं किए हैं और निरीक्षण के लिए जबरदस्त'
'Garbage dump all over'
'Puttenahalli jheel road jo Arekere ki or jaati hai, itni kharab hai ki lagatar jaam lagte rehte hain kyuki koi fisal kar gir gaya ya vaahan bahut dheere chal rahe hain. Aur agar bhari barish hoti hai,'
'Desilted Mud is not cleared making the road difficult to drive,'

Prediction Pool Examples:
'Sir, we are facing lot of water problem in 8th main byrappa layout s g palya, c v Raman Nagar Bangalore 93.\nI 

# 2. Text Cleaning Function

This section defines a reusable text cleaning function for both
the supervised dataset and the prediction pool.

The cleaning process focuses on HTML entity decoding, Unicode
normalization and whitespace standardization while preserving
multilingual scripts, numbers, punctuation and the original meaning.

In [3]:
# Define a reusable function for cleaning complaint text
def clean_text(text):

    # Handle missing or non-string values safely
    if pd.isna(text):
        return ""

    # Convert the input into a string
    text = str(text)

    # Decode HTML entities such as &amp; and &nbsp;
    text = html.unescape(text)

    # Normalize Unicode representations without removing language characters
    text = unicodedata.normalize("NFC", text)

    # Replace line breaks, tabs and repeated whitespace with a single space
    text = re.sub(r"\s+", " ", text)

    # Remove unnecessary whitespace from the beginning and end
    text = text.strip()

    # Return the cleaned complaint text
    return text

In [4]:
# Test the cleaning function using different text formats
test_samples = [
    "Garbage dumped here\n\nPlease clean it.",
    "Road &amp; drainage problem",
    "सीएमसी का पानी नहीं आ रहा है।",
    "Kachra kachra kachra",
    "Street lights are not working!!!"
]

# Display original and cleaned versions for comparison
for sample in test_samples:

    print("Original:", repr(sample))
    print("Cleaned :", repr(clean_text(sample)))
    print("-" * 80)

Original: 'Garbage dumped here\n\nPlease clean it.'
Cleaned : 'Garbage dumped here Please clean it.'
--------------------------------------------------------------------------------
Original: 'Road &amp; drainage problem'
Cleaned : 'Road & drainage problem'
--------------------------------------------------------------------------------
Original: 'सीएमसी का पानी नहीं आ रहा है।'
Cleaned : 'सीएमसी का पानी नहीं आ रहा है।'
--------------------------------------------------------------------------------
Original: 'Kachra kachra kachra'
Cleaned : 'Kachra kachra kachra'
--------------------------------------------------------------------------------
Original: 'Street lights are not working!!!'
Cleaned : 'Street lights are not working!!!'
--------------------------------------------------------------------------------


# 3. Apply Text Cleaning

Apply the validated cleaning function to the supervised dataset
and the unlabelled prediction pool.

The original text columns will remain unchanged, while cleaned
versions will be stored in separate columns.

In [5]:
# Create cleaned text for the supervised dataset
supervised_df["cleaned_text"] = (
    supervised_df["model_text"].apply(clean_text)
)

# Create cleaned text for the prediction pool
prediction_df["cleaned_text"] = (
    prediction_df["description"].apply(clean_text)
)

# Display the updated dimensions of both datasets
print("Supervised Dataset Shape:", supervised_df.shape)
print("Prediction Pool Shape:", prediction_df.shape)

# Display original and cleaned text side by side
display(
    supervised_df[
        ["model_text", "cleaned_text", "target_category"]
    ].head(10)
)

display(
    prediction_df[
        ["description", "cleaned_text", "detectedLanguage"]
    ].head(10)
)

Supervised Dataset Shape: (16519, 4)
Prediction Pool Shape: (27686, 14)


,model_text,cleaned_text,target_category
0,Sadak paar karne wala concrete ka puliya adhur...,Sadak paar karne wala concrete ka puliya adhur...,Road Infrastructure
1,"सीएमसी का पानी जारी नहीं किया जा रहा है, वे बा...","सीएमसी का पानी जारी नहीं किया जा रहा है, वे बा...",Water Supply
2,Garbage dump all over,Garbage dump all over,Waste Management
3,Puttenahalli jheel road jo Arekere ki or jaati...,Puttenahalli jheel road jo Arekere ki or jaati...,Road Infrastructure
4,Desilted Mud is not cleared making the road di...,Desilted Mud is not cleared making the road di...,Waste Management
5,Sewage ka paani naali mein pravesh kar raha hai.,Sewage ka paani naali mein pravesh kar raha hai.,Drainage and Sewerage
6,"Bellandur mein, Ecospace Tech Park ke paas, ek...","Bellandur mein, Ecospace Tech Park ke paas, ek...",Road Infrastructure
7,यहाँ पानी की उचित व्यवस्था नहीं है। हमें दो दि...,यहाँ पानी की उचित व्यवस्था नहीं है। हमें दो दि...,Water Supply
8,Sadak ke kinare har jagah kachra dala ja raha ...,Sadak ke kinare har jagah kachra dala ja raha ...,Waste Management
9,The road opposite to the cantonment railway st...,The road opposite to the cantonment railway st...,Road Infrastructure


,description,cleaned_text,detectedLanguage
0,"Sir, we are facing lot of water problem in 8th...","Sir, we are facing lot of water problem in 8th...",English
1,"Doddabommasandra Arch se Nanjappa Circle, Nanj...","Doddabommasandra Arch se Nanjappa Circle, Nanj...",Hinglish
2,Kachra kachra kachra,Kachra kachra kachra,Hinglish
3,Garbage dumped next to mysore circle bus stop....,Garbage dumped next to mysore circle bus stop....,English
4,Too many pot holes near GIIS school and petrol...,Too many pot holes near GIIS school and petrol...,English
5,"Akshay Nagar Yelenahalli Bangalore 560068, 9th...","Akshay Nagar Yelenahalli Bangalore 560068, 9th...",Hinglish
6,Ek car 10 dinon se zyada samay se hamare ghar ...,Ek car 10 dinon se zyada samay se hamare ghar ...,Hinglish
7,सड़क क्षतिग्रस्त हो रही है।,सड़क क्षतिग्रस्त हो रही है।,Hindi
8,"Namaste ShriMan/Shrimati, kripya is par kuch k...","Namaste ShriMan/Shrimati, kripya is par kuch k...",Hinglish
9,No parking leads to traffic jam,No parking leads to traffic jam,English


# 4. Before and After Comparison

Compare original and cleaned complaint texts to verify that
unnecessary formatting has been standardized without losing
important multilingual content.

In [6]:
# Compare original and cleaned character lengths
supervised_df["original_length"] = (
    supervised_df["model_text"].str.len()
)

supervised_df["cleaned_length"] = (
    supervised_df["cleaned_text"].str.len()
)

prediction_df["original_length"] = (
    prediction_df["description"].str.len()
)

prediction_df["cleaned_length"] = (
    prediction_df["cleaned_text"].str.len()
)

# Display descriptive statistics for the supervised dataset
display(
    supervised_df[
        ["original_length", "cleaned_length"]
    ].describe().round(2)
)

# Display descriptive statistics for the prediction pool
display(
    prediction_df[
        ["original_length", "cleaned_length"]
    ].describe().round(2)
)

# Count records whose text changed during cleaning
print(
    "Supervised records changed:",
    (supervised_df["model_text"] != supervised_df["cleaned_text"]).sum()
)

print(
    "Prediction records changed:",
    (prediction_df["description"] != prediction_df["cleaned_text"]).sum()
)

,original_length,cleaned_length
count,16519.00,16519.00
mean,264.96,264.30
std,221.49,220.63
min,10.00,10.00
25%,129.00,129.00
50%,201.00,201.00
75%,332.50,332.00
max,3778.00,3774.00


,original_length,cleaned_length
count,27686.00,27686.00
mean,139.41,139.17
std,65.80,65.63
min,7.00,7.00
25%,78.00,78.00
50%,165.00,165.00
75%,200.00,200.00
max,200.00,203.00


Supervised records changed: 5219
Prediction records changed: 6093


In [7]:
# Identify prediction records whose cleaned text is longer than the original
length_increased = prediction_df[
    prediction_df["cleaned_length"] > prediction_df["original_length"]
].copy()

# Display the number of affected records
print("Records with increased text length:", len(length_increased))

# Display original and cleaned text for inspection
display(
    length_increased[
        [
            "description",
            "cleaned_text",
            "original_length",
            "cleaned_length"
        ]
    ].head(15)
)

# Calculate the maximum increase in character length
if not length_increased.empty:
    print(
        "Maximum length increase:",
        (
            length_increased["cleaned_length"]
            - length_increased["original_length"]
        ).max()
    )

Records with increased text length: 96


,description,cleaned_text,original_length,cleaned_length
126,एयरपोर्ट वॉल रोड पर एक साल पहले कुछ मैनहोल लगा...,एयरपोर्ट वॉल रोड पर एक साल पहले कुछ मैनहोल लगा...,200,201
337,कई बार शिकायत करने के बावजूद ठेकेदार और बीबीएम...,कई बार शिकायत करने के बावजूद ठेकेदार और बीबीएम...,200,201
737,नाली का पानी सड़क पर आ गया है और बहुत ज़्यादा प...,नाली का पानी सड़क पर आ गया है और बहुत ज़्यादा ...,200,202
1001,पिछले 2 हफ़्तों से सड़क की मरम्मत नहीं की जा रह...,पिछले 2 हफ़्तों से सड़क की मरम्मत नहीं की जा र...,167,168
1258,यह मोटरसाइकिल कई हफ़्तों से एक ही जगह पर गैरकान...,यह मोटरसाइकिल कई हफ़्तों से एक ही जगह पर गैरका...,200,201
1343,"नमस्ते,\nयह जगह एक पब के ठीक सामने है, और यहाँ...","नमस्ते, यह जगह एक पब के ठीक सामने है, और यहाँ ...",200,202
1614,पिछले 3 हफ़्तों से पानी की आपूर्ति नहीं है...यह...,पिछले 3 हफ़्तों से पानी की आपूर्ति नहीं है...य...,65,66
1807,कमराज रोड पर एक्सिस बैंक एटीएम के पास मच्छरों ...,कमराज रोड पर एक्सिस बैंक एटीएम के पास मच्छरों ...,78,79
1920,बाउंस टो सर्विस की पांच से ज़्यादा लॉरियां एक ह...,बाउंस टो सर्विस की पांच से ज़्यादा लॉरियां एक ...,79,82
2032,मैं छठी क्रॉस में रहता हूँ और शुरू में एक महिल...,मैं छठी क्रॉस में रहता हूँ और शुरू में एक महिल...,200,201


Maximum length increase: 3


# 5. Text Cleaning Validation

This section validates the cleaned datasets by checking missing
values, empty texts, duplicate records, and changes in text length.

The objective is to ensure that preprocessing has preserved the
dataset structure and meaningful complaint content.

In [8]:
# Check missing values in the cleaned supervised dataset
print("Supervised Dataset")
print("Missing cleaned texts:",
      supervised_df["cleaned_text"].isna().sum())

# Check missing values in the cleaned prediction pool
print("\nPrediction Pool")
print("Missing cleaned texts:",
      prediction_df["cleaned_text"].isna().sum())

# Check whether any cleaned complaint became empty
print("\nEmpty cleaned texts:")
print("Supervised:",
      (supervised_df["cleaned_text"].str.len() == 0).sum())

print("Prediction:",
      (prediction_df["cleaned_text"].str.len() == 0).sum())

# Verify that the original number of records is preserved
print("\nRecord Count Validation:")
print("Supervised:", len(supervised_df) == 16519)
print("Prediction:", len(prediction_df) == 27686)

# Check whether target labels remain unchanged
print("\nTarget category count:",
      supervised_df["target_category"].nunique())

# Check whether source information remains unchanged
print("\nSource dataset distribution:")
print(supervised_df["source_dataset"].value_counts())

Supervised Dataset
Missing cleaned texts: 0

Prediction Pool
Missing cleaned texts: 0

Empty cleaned texts:
Supervised: 0
Prediction: 0

Record Count Validation:
Supervised: True
Prediction: True

Target category count: 12

Source dataset distribution:
source_dataset
Dataset_B_English         15296
Dataset_A_Multilingual     1223
Name: count, dtype: int64


In [9]:
# Check whether any cleaned texts are duplicated
print("Duplicate cleaned texts in supervised dataset:",
      supervised_df["cleaned_text"].duplicated().sum())

print("Duplicate cleaned texts in prediction pool:",
      prediction_df["cleaned_text"].duplicated().sum())

# Count records where the cleaned text differs from the original
print("\nChanged text records:")
print("Supervised:",
      (supervised_df["model_text"] != supervised_df["cleaned_text"]).sum())

print("Prediction:",
      (prediction_df["description"] != prediction_df["cleaned_text"]).sum())

# Verify that all original text columns remain unchanged
print("\nOriginal text preservation:")
print("Supervised:",
      supervised_df["model_text"].equals(
          pd.read_csv("../data/interim/civicsense_supervised_dataset.csv")["model_text"]
      ))

print("Prediction:",
      prediction_df["description"].equals(
          pd.read_csv("../data/interim/civicsense_prediction_pool.csv")["description"]
      ))

Duplicate cleaned texts in supervised dataset: 0
Duplicate cleaned texts in prediction pool: 880

Changed text records:
Supervised: 5219
Prediction: 6093

Original text preservation:
Supervised: True
Prediction: True


In [10]:
# Count duplicates in the original prediction descriptions
original_duplicates = prediction_df["description"].duplicated().sum()

# Count duplicates after text cleaning
cleaned_duplicates = prediction_df["cleaned_text"].duplicated().sum()

# Identify records that were unique originally but became duplicates after cleaning
original_unique_mask = ~prediction_df["description"].duplicated()
cleaned_duplicate_mask = prediction_df["cleaned_text"].duplicated(keep=False)

# Identify cleaned duplicate groups containing originally different texts
new_duplicate_records = prediction_df[
    original_unique_mask & cleaned_duplicate_mask
].copy()

# Display duplicate counts before and after cleaning
print("Original description duplicates:", original_duplicates)
print("Cleaned text duplicates:", cleaned_duplicates)
print("Originally unique records in cleaned duplicate groups:",
      len(new_duplicate_records))

# Inspect examples of newly formed duplicate groups
display(
    new_duplicate_records[
        ["description", "cleaned_text"]
    ].head(15)
)

Original description duplicates: 874
Cleaned text duplicates: 880
Originally unique records in cleaned duplicate groups: 387


,description,cleaned_text
22,Doresinipalya,Doresinipalya
77,Footpath ka rakhrakhav,Footpath ka rakhrakhav
85,Require new footpath,Require new footpath
106,Respected Sir / Madam.\n\nThis is to address t...,Respected Sir / Madam. This is to address the ...
131,Debris on footpath,Debris on footpath
175,फुटपाथ पर मलबा,फुटपाथ पर मलबा
176,Very Very Bad Road and needs to be retarded ur...,Very Very Bad Road and needs to be retarded ur...
234,Street light kaam nahi kar rahi hai.,Street light kaam nahi kar rahi hai.
235,Street light not working,Street light not working
237,Kindly clear this area as it encourages passer...,Kindly clear this area as it encourages passer...


In [11]:
# Remove exact original description duplicates for this comparison
unique_original_texts = (
    prediction_df[
        ["description", "cleaned_text"]
    ]
    .drop_duplicates(subset=["description"])
)

# Group originally unique descriptions by their cleaned text
collision_groups = (
    unique_original_texts
    .groupby("cleaned_text")
    .agg(
        original_variants=("description", "nunique"),
        original_examples=("description", list)
    )
    .reset_index()
)

# Keep only groups where different original texts became identical
collision_groups = collision_groups[
    collision_groups["original_variants"] > 1
].copy()

# Display the number of newly merged text groups
print("New collision groups:", len(collision_groups))

# Calculate how many distinct original variants were merged
print(
    "Distinct original text variants merged:",
    (collision_groups["original_variants"] - 1).sum()
)

# Inspect the examples
display(collision_groups.head(15))

New collision groups: 6
Distinct original text variants merged: 6


,cleaned_text,original_variants,original_examples
221,2 street lights are not working in BK Residenc...,2,[2 street lights are not working in BK Residen...
6658,In my locality there's a house which has illeg...,2,[In my locality there's a house which has ille...
12536,Road uneven and potholes Near Adigas B G Road ...,2,[Road uneven and potholes Near Adigas B G Road...
13850,"Sir, With all due respect, I am writing on beh...",2,"[Sir,\nWith all due respect, I am writing on b..."
15247,The residents of 6th A main and 13th cross wou...,2,[The residents of 6th A main and 13th cross wo...
18925,एक हफ़्ते से स्ट्रीट लाइट काम नहीं कर रही है।,2,"[एक हफ़्ते से स्ट्रीट लाइट काम नहीं कर रही है।,..."


# 6. Final Dataset Preparation and Saving

This section prepares the final processed datasets for subsequent
tokenization and model development.

The original text columns, target labels and source information
are preserved. Cleaned complaint text is stored separately.

Duplicate complaint records are retained in the prediction pool
because each record represents an individual complaint.

In [12]:
# Create the final supervised dataset with cleaned text and labels
final_supervised_df = supervised_df[
    ["cleaned_text", "target_category", "source_dataset"]
].copy()

# Create the final prediction dataset while retaining all original metadata
final_prediction_df = prediction_df.copy()

# Remove temporary analysis columns from the prediction dataset
final_prediction_df = final_prediction_df.drop(
    columns=["original_length", "cleaned_length"]
)

# Display the final dataset dimensions
print("Final Supervised Dataset:", final_supervised_df.shape)
print("Final Prediction Dataset:", final_prediction_df.shape)

# Inspect the final supervised dataset
display(final_supervised_df.head())

# Inspect the final prediction dataset
display(
    final_prediction_df[
        ["description", "cleaned_text", "detectedLanguage"]
    ].head()
)

Final Supervised Dataset: (16519, 3)
Final Prediction Dataset: (27686, 14)


,cleaned_text,target_category,source_dataset
0,Sadak paar karne wala concrete ka puliya adhur...,Road Infrastructure,Dataset_A_Multilingual
1,"सीएमसी का पानी जारी नहीं किया जा रहा है, वे बा...",Water Supply,Dataset_A_Multilingual
2,Garbage dump all over,Waste Management,Dataset_A_Multilingual
3,Puttenahalli jheel road jo Arekere ki or jaati...,Road Infrastructure,Dataset_A_Multilingual
4,Desilted Mud is not cleared making the road di...,Waste Management,Dataset_A_Multilingual


,description,cleaned_text,detectedLanguage
0,"Sir, we are facing lot of water problem in 8th...","Sir, we are facing lot of water problem in 8th...",English
1,"Doddabommasandra Arch se Nanjappa Circle, Nanj...","Doddabommasandra Arch se Nanjappa Circle, Nanj...",Hinglish
2,Kachra kachra kachra,Kachra kachra kachra,Hinglish
3,Garbage dumped next to mysore circle bus stop....,Garbage dumped next to mysore circle bus stop....,English
4,Too many pot holes near GIIS school and petrol...,Too many pot holes near GIIS school and petrol...,English


In [13]:
from pathlib import Path

# Define new output paths inside the interim data directory
supervised_output = Path(
    "../data/interim/civicsense_supervised_cleaned.csv"
)

prediction_output = Path(
    "../data/interim/civicsense_prediction_cleaned.csv"
)

# Prevent accidental overwriting of existing files
if supervised_output.exists() or prediction_output.exists():
    raise FileExistsError(
        "An output file already exists. Check the files before saving."
    )

# Save the processed supervised dataset
final_supervised_df.to_csv(
    supervised_output,
    index=False,
    encoding="utf-8-sig"
)

# Save the processed prediction dataset
final_prediction_df.to_csv(
    prediction_output,
    index=False,
    encoding="utf-8-sig"
)

# Confirm the saved file locations
print("Supervised dataset saved:", supervised_output)
print("Prediction dataset saved:", prediction_output)

Supervised dataset saved: ..\data\interim\civicsense_supervised_cleaned.csv
Prediction dataset saved: ..\data\interim\civicsense_prediction_cleaned.csv


In [14]:
# Reload both saved datasets to verify their dimensions
supervised_check = pd.read_csv(supervised_output)
prediction_check = pd.read_csv(prediction_output)

# Verify that the expected number of records is preserved
assert supervised_check.shape[0] == 16519
assert prediction_check.shape[0] == 27686

# Verify that the cleaned text columns exist
assert "cleaned_text" in supervised_check.columns
assert "cleaned_text" in prediction_check.columns

# Verify that no cleaned text is missing
assert supervised_check["cleaned_text"].notna().all()
assert prediction_check["cleaned_text"].notna().all()

print("Supervised dataset: VERIFIED")
print("Prediction dataset: VERIFIED")
print("Both processed datasets saved successfully.")

Supervised dataset: VERIFIED
Prediction dataset: VERIFIED
Both processed datasets saved successfully.
